## 1、导入相关函数
pip install opencv-python insightface onnxruntime tensorflow numpy pandas scikit-learn pillow flask mysql-connector-python

insightface


In [1]:
#人脸识别相关函数
import os
import cv2
import io
import insightface
import numpy as np
from sklearn import preprocessing
from flask import Flask, request, jsonify
from PIL import Image
import warnings
import mysql.connector
import pandas as pd
import pickle
from tensorflow.keras.models import load_model

warnings.filterwarnings("ignore")

c:\Program Files\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Program Files\Python\Python310\lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.6' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


# 2、人脸识别类

In [2]:
# 封装了人脸识别的主要功能，包括初始化、加载人脸库、识别、注册和检测
class FaceRecognition:
    # 初始化：初始化人脸识别模型并加载人脸库
    def __init__(self, gpu_id=0, face_db='face_db', threshold=1.24, det_thresh=0.50, det_size=(640, 640)):
        self.gpu_id = gpu_id                    # 指定使用哪个 GPU（默认 0）
        self.face_db = face_db                  # 人脸库路径（默认 "face_db"）
        self.threshold = threshold              # 人脸特征匹配的阈值（默认 1.24）
        self.det_thresh = det_thresh            # 人脸检测置信度阈值（默认 0.50）
        self.det_size = det_size                # 检测输入图像尺寸（默认 640x640）
        # 初始化 insightface 的 FaceAnalysis 模型，使用 CUDA加速
        self.model = insightface.app.FaceAnalysis(root=r'C:\Program Files\Python\Python310\Lib\site-packages\insightface', allowed_modules=None, providers=['CUDAExecutionProvider'])
        self.model.prepare(ctx_id=self.gpu_id, det_thresh=self.det_thresh, det_size=self.det_size)
        self.faces_embedding = list()
        # 调用 load_faces 方法加载人脸库中的已有数据
        self.load_faces(self.face_db)

    # 加载人脸库中的人脸
    def load_faces(self, face_db_path):
        # 检查人脸库路径是否存在，不存在则创建
        if not os.path.exists(face_db_path):
            os.makedirs(face_db_path)
        # 遍历文件夹中的所有文件
        for root, dirs, files in os.walk(face_db_path):
            for file in files:
                # 使用 cv2.imdecode 读取图像
                input_image = cv2.imdecode(np.fromfile(os.path.join(root, file), dtype=np.uint8), 1)
                if input_image is None:
                    print(f"Warning: Unable to read {file}. Skipping this file.")
                    continue
                user_name = file.split(".")[0]
                # 使用 model.get 检测人脸
                faces = self.model.get(input_image)
                if len(faces) == 0:
                    print(f"Warning: No face detected in {file}. Skipping this file.")
                    continue
                # 如果检测到人脸，提取特征向量（embedding），标准化后存储到 faces_embedding 列表中
                face = faces[0]
                embedding = np.array(face.embedding).reshape((1, -1))
                embedding = preprocessing.normalize(embedding)
                self.faces_embedding.append({
                    "user_name": user_name,               # 文件名（用户名）
                    "feature": embedding                  # 标准化后的特征向量
                })
    
    # 特征比较：比较两个特征向量的相似度
    @staticmethod
    def feature_compare(feature1, feature2, threshold):
        # 计算两个特征向量的差值
        diff = np.subtract(feature1, feature2)
        # 计算差值的平方和（欧几里得距离的平方）
        dist = np.sum(np.square(diff), 1)
        return dist < threshold

    # 人脸注册：将新的人脸添加到人脸库（未用）
    def register(self, image, user_name):
        # 使用 model.get 检测图像中的人脸
        faces = self.model.get(image)
        if len(faces) != 1:
            return '图片检测不到人脸'
        # 提取特征向量并标准化
        embedding = np.array(faces[0].embedding).reshape((1, -1))
        embedding = preprocessing.normalize(embedding)
        # 检查该特征是否已存在于人脸库中
        is_exits = False
        for com_face in self.faces_embedding:
            r = self.feature_compare(embedding, com_face["feature"], self.threshold)
            if r:
                is_exits = True
        if is_exits:
            return '该用户已存在'
        # 符合注册条件保存图片
        cv2.imencode('.jpg', image)[1].tofile(os.path.join(self.face_db, '%s.jpg' % user_name))
        # 把特征添加到人脸特征库中
        self.faces_embedding.append({
            "user_name": user_name,
            "feature": embedding
        })
        return "success"
    

    # 人脸识别：对输入图像进行人脸识别，返回人脸名称和边界框
    def recognition(self, image):
        # 使用 model.get 检测图像中的所有人脸
        faces = self.model.get(image)
        results = list()
        for face in faces:
            # 提取特征向量并标准化
            embedding = np.array(face.embedding).reshape((1, -1))
            embedding = preprocessing.normalize(embedding)
            # 与人脸库中的特征逐一比较
            user_name = "unknown"
            for com_face in self.faces_embedding:
                r = self.feature_compare(embedding, com_face["feature"], self.threshold)
                if r:
                    user_name = com_face["user_name"]
            results.append((user_name, face.bbox))
        return results


    # 人脸检测：检测图像中的人脸并返回详细信息（包括边界框、关键点、性别、年龄等）
    def detect(self, image):
        # 使用 model.get 检测人脸
        faces = self.model.get(image)
        results = list()
        for face in faces:
            # 提取属性（如边界框 bbox、关键点 kps、置信度 det_score、性别 gender、年龄 age、特征 embedding）
            result = dict()
            result["bbox"] = np.array(face.bbox).astype(np.int32)
            result["kps"] = np.array(face.kps).astype(np.int32).tolist()
            result["det_score"] = float(face.det_score)
            if face.gender is not None:
                result["gender"] = face.gender
            if face.age is not None:
                result["age"] = face.age
            if face.embedding is not None:
                result["embedding"] = face.embedding
            # 如果有特征向量，进行人脸识别，匹配人脸库中的名称
            if face.embedding is not None:
                embedding = np.array(face.embedding).reshape((1, -1))
                embedding = preprocessing.normalize(embedding)
                user_name = "unknown"
                for com_face in self.faces_embedding:
                    if self.feature_compare(embedding, com_face["feature"], self.threshold):
                        user_name = com_face["user_name"]
                        break
                result["name"] = user_name
            else:
                result["name"] = "unknown"
            results.append(result)
        return results
detector = FaceRecognition()

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Program Files\Python\Python310\Lib\site-packages\insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Program Files\Python\Python310\Lib\site-packages\insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Program Files\Python\Python310\Lib\site-packages\insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Program Files\Python\Python310\Lib\site-packages\insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with opti

# 3、辅助函数

In [3]:
# 辅助函数：从 detect 的结果中提取关键信息（置信度、名称、关键点、性别）
def face_r(frame):
    out=[]
    # 获取检测结果
    detection_results = detector.detect(frame)[:1]
    for result in detection_results:
        bbox = result["bbox"]
        kps = result["kps"][:2]
        det_score = result["det_score"]
        name = result["name"]
        out.append(det_score)
        out.append(name)
        out.append(kps)
        out.append(result["gender"])
    return out



# 数据转换函数：将numpy类型转换为标准 Python 类型，以便 JSON 序列化
# 递归处理输入（支持字典、列表、numpy 数组等）。将 np.ndarray 转为列表，np.int64 转为 int，np.float64 转为 float。其他类型保持不变
def convert_results(results):
    if isinstance(results, dict):
        return {key: convert_results(value) for key, value in results.items()}
    elif isinstance(results, list):
        return [convert_results(item) for item in results]
    elif isinstance(results, np.ndarray):
        return results.tolist()
    elif isinstance(results, (np.int64, np.float64)):
        return float(results) if isinstance(results, np.float64) else int(results)
    return results

# 人脸验证接口

In [ ]:
app = Flask(__name__)

@app.route('/detect_faceid', methods=['POST'])
def detect_faceid():
    if 'image' not in request.files:
        return jsonify({'error': '未提供图片或者图片格式错误'}), 400
    # 使用 PIL 读取上传的图像并转为 numpy 数组
    file = request.files['image']
    image = Image.open(io.BytesIO(file.read()))
    image_np = np.array(image)
    # 调用 face_r 获取识别结果
    results = face_r(image_np)
    # 使用 convert_results 转换结果格式
    results = convert_results(results)
    # 以 JSON 格式返回结果
    return jsonify({'results': results})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.101:5000
Press CTRL+C to quit


127.0.0.1 - - [12/May/2025 09:57:32] "POST /detect_faceid HTTP/1.1" 200 -
